# Round 3 notebook 00 — data foundation

        Purpose: establish the exact replayed dataset, product set, trade counts, timestamp structure,
        and the basic markout helpers used by the later notebooks.

        Principle: before fitting anything, make the raw market mechanics auditable.

In [1]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

from pathlib import Path
import math
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

DATA = Path("data/ROUND3")
DAYS = [0, 1, 2]

def load_round3():
    prices = []
    trades = []
    for day in DAYS:
        p = pd.read_csv(DATA / f"prices_round_3_day_{day}.csv", sep=";")
        t = pd.read_csv(DATA / f"trades_round_3_day_{day}.csv", sep=";")
        p["day"] = day
        t["day"] = day
        prices.append(p)
        trades.append(t)
    prices = pd.concat(prices, ignore_index=True)
    trades = pd.concat(trades, ignore_index=True)
    for col in prices.columns:
        if col.startswith(("bid_", "ask_")) or col == "mid_price":
            prices[col] = pd.to_numeric(prices[col], errors="coerce")
    trades["price"] = pd.to_numeric(trades["price"], errors="coerce")
    trades["quantity"] = pd.to_numeric(trades["quantity"], errors="coerce")
    return prices, trades

def add_book_features(prices):
    p = prices.copy()
    p["best_bid"] = p["bid_price_1"]
    p["best_ask"] = p["ask_price_1"]
    p["spread"] = p["best_ask"] - p["best_bid"]
    bid_cols = ["bid_price_1", "bid_price_2", "bid_price_3"]
    ask_cols = ["ask_price_1", "ask_price_2", "ask_price_3"]
    p["wall_bid"] = p[bid_cols].min(axis=1)
    p["wall_ask"] = p[ask_cols].max(axis=1)
    p["wall_mid"] = (p["wall_bid"] + p["wall_ask"]) / 2
    p["wall_spread"] = p["wall_ask"] - p["wall_bid"]
    p["n_bid_levels"] = p[bid_cols].notna().sum(axis=1)
    p["n_ask_levels"] = p[ask_cols].notna().sum(axis=1)
    p["mid_minus_wall"] = p["mid_price"] - p["wall_mid"]
    return p

def add_future_mid(prices, horizons=(1, 5, 10, 25, 50, 100, 200, 500, 1000, 2000, 5000)):
    p = prices.sort_values(["product", "day", "timestamp"]).copy()
    for h in horizons:
        p[f"fut_mid_{h}"] = p.groupby(["product", "day"])["mid_price"].shift(-h)
        p[f"ret_{h}"] = p[f"fut_mid_{h}"] - p["mid_price"]
    return p

def attach_trades(prices, trades):
    t = trades.merge(
        prices,
        left_on=["day", "timestamp", "symbol"],
        right_on=["day", "timestamp", "product"],
        how="left",
    )
    t["side"] = np.where(
        t["price"] == t["ask_price_1"],
        "buy_agg",
        np.where(t["price"] == t["bid_price_1"], "sell_agg", "other"),
    )
    return t

def maker_markout_table(trades_with_book, horizons=(1, 10, 50, 100, 200, 500, 1000, 2000, 5000)):
    t = trades_with_book.copy()
    for h in horizons:
        # PnL for a passive maker who sold to an aggressive buyer, or bought from an aggressive seller.
        t[f"mk_{h}"] = np.where(
            t["side"] == "buy_agg",
            t["price"] - t[f"fut_mid_{h}"],
            np.where(t["side"] == "sell_agg", t[f"fut_mid_{h}"] - t["price"], np.nan),
        )
    agg = {
        "n": ("price", "size"),
        "avg_qty": ("quantity", "mean"),
        "avg_price": ("price", "mean"),
    }
    for h in horizons:
        agg[f"mk_{h}"] = (f"mk_{h}", "mean")
    return t.groupby(["symbol", "side"]).agg(**agg).round(3).reset_index()

def slow_ema_payoff(prices, span=1000, horizons=(10, 25, 50, 100, 200, 500, 1000, 2000)):
    rows = []
    p = prices.sort_values(["product", "day", "timestamp"]).copy()
    alpha = 2 / (span + 1)
    p["ema"] = p.groupby(["product", "day"])["mid_price"].transform(
        lambda s: s.ewm(alpha=alpha, adjust=False).mean()
    )
    p["dev"] = p["mid_price"] - p["ema"]
    for product, g in p.groupby("product"):
        if g["mid_price"].std() == 0:
            rows.append({"product": product, "dev_q10": 0, "dev_q90": 0, "best_H": None, "edge_ticks": 0})
            continue
        lo, hi = g["dev"].quantile(0.10), g["dev"].quantile(0.90)
        low = g[g["dev"] <= lo]
        high = g[g["dev"] >= hi]
        best = None
        for h in horizons:
            low_ret = low.groupby("day")["mid_price"].shift(-h) if False else low[f"ret_{h}"]
            high_ret = high[f"ret_{h}"]
            edge = (low_ret.mean() - high_ret.mean()) / 2
            rec = (h, edge, low_ret.mean(), high_ret.mean())
            if best is None or edge > best[1]:
                best = rec
        rows.append({
            "product": product,
            "dev_q10": lo,
            "dev_q90": hi,
            "best_H": best[0],
            "edge_ticks": best[1],
            "low_future_move": best[2],
            "high_future_move": best[3],
        })
    return pd.DataFrame(rows).sort_values("edge_ticks", ascending=False).round(3)

In [2]:
prices_raw, trades_raw = load_round3()
prices = add_future_mid(add_book_features(prices_raw))
trades = attach_trades(prices, trades_raw)

display(Markdown("## Dataset shape"))
display(pd.DataFrame({
    "object": ["price rows", "trade rows", "unique products", "days"],
    "count": [len(prices), len(trades_raw), prices["product"].nunique(), prices["day"].nunique()],
}))

display(Markdown("## Product row counts"))
display(prices["product"].value_counts().sort_index().to_frame("price_rows"))

display(Markdown("## Trade counts and side classification"))
side_counts = pd.crosstab(trades["symbol"], trades["side"])
side_counts["total"] = side_counts.sum(axis=1)
display(side_counts.sort_index())

display(Markdown("## Timestamp structure"))
ts = prices.groupby("day")["timestamp"].agg(["min", "max", "nunique"]).reset_index()
ts["step_median"] = prices.groupby("day")["timestamp"].apply(lambda s: s.sort_values().drop_duplicates().diff().median()).values
display(ts)

## Dataset shape

,object,count
0,price rows,360000
1,trade rows,4048
2,unique products,12
3,days,3


## Product row counts

,price_rows
product,
HYDROGEL_PACK,30000
VELVETFRUIT_EXTRACT,30000
VEV_4000,30000
VEV_4500,30000
VEV_5000,30000
VEV_5100,30000
VEV_5200,30000
VEV_5300,30000
VEV_5400,30000


## Trade counts and side classification

side,buy_agg,other,sell_agg,total
symbol,,,,
HYDROGEL_PACK,524,0,486,1010
VELVETFRUIT_EXTRACT,777,6,589,1372
VEV_4000,226,0,238,464
VEV_4500,1,0,0,1
VEV_5000,1,0,0,1
VEV_5100,1,0,0,1
VEV_5200,1,0,17,18
VEV_5300,1,1,119,121
VEV_5400,0,0,225,225


## Timestamp structure

,day,min,max,nunique,step_median
0,0,0,999900,10000,100.0
1,1,0,999900,10000,100.0
2,2,0,999900,10000,100.0


## Immediate read

        - There are 12 R3 symbols: two delta-1 products, eight active vouchers, and two pinned far-OTM vouchers.
        - Each product has 10,000 rows per historical day, with timestamps `0..999900` in increments of 100.
        - Trade names are redacted, so bot analysis must be inferred from trade fingerprints and book structure.